# FaceRestore Training — Kaggle T4

## Before Running
1. **Accelerator**: GPU T4 x2 (Kaggle only offers T4x2 or P100)
2. **Internet**: ON
3. **Add Data**: Attach your 3 datasets
4. **Secrets**: Add `WANDB_API_KEY` (optional)

## Use "Save & Run All" for training
Interactive mode output is **lost** if session stops.
Always use **Save Version → Save & Run All** so checkpoints are preserved.

## What You Get vs Colab
- 12hr sessions (vs 3.5hr) → **3 epochs per session**
- 30 GPU-hrs/week → ~8 epochs per account per week
- No anti-idle hack needed
- Close browser safely — training continues
- W&B tracks across all sessions/accounts


In [ ]:
# Cell 1: Install Dependencies + W&B Auth
import os

!pip install -q jsonlines tqdm "matplotlib<3.8" wandb tensorboard

# Kaggle secrets need special API
wandb_key = None
try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    wandb_key = os.environ.get('WANDB_API_KEY')

try:
    import wandb
    if wandb_key:
        wandb.login(key=wandb_key)
        print('W&B: Authenticated via Kaggle secret')
    else:
        print('W&B: No API key found — skipping (TensorBoard still works)')
except Exception as e:
    print(f'W&B: Skipped ({e})')

print('\nCell 1 done.')


In [ ]:
# Cell 2: Setup Project Structure & Prepare Data
import os, shutil

KAGGLE_INPUT = '/kaggle/input'
WORK_DIR     = '/kaggle/working'
PROJECT_DIR  = os.path.join(WORK_DIR, 'inpaint_project')
DATASET_DIR  = os.path.join(WORK_DIR, 'dataset')
CKPT_DIR     = os.path.join(WORK_DIR, 'facerestore_checkpoints')
TB_LOG_DIR   = os.path.join(WORK_DIR, 'logs')

def find_file(root, name):
    for dirpath, _, filenames in os.walk(root):
        if name in filenames:
            return os.path.join(dirpath, name)
    return None

# --- 1. Training code ---
if not os.path.exists(PROJECT_DIR):
    module_path = find_file(KAGGLE_INPUT, 'module.py')
    if module_path and 'inpaint' in os.path.dirname(module_path):
        train_root = os.path.dirname(os.path.dirname(module_path))
        shutil.copytree(train_root, PROJECT_DIR)
        print(f'Training code: {train_root}')
    else:
        print('ERROR: Training code not found!')
else:
    print(f'Training code already at {PROJECT_DIR}')

# --- 2. Face images ---
faces_dir = os.path.join(DATASET_DIR, 'faces')
os.makedirs(faces_dir, exist_ok=True)
celeba_found = len(os.listdir(faces_dir)) > 1000

if not celeba_found:
    for dirpath, _, filenames in os.walk(KAGGLE_INPUT):
        jpgs = [f for f in filenames if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if len(jpgs) > 1000:
            print(f'Found {len(jpgs)} face images in {dirpath}')
            print('Creating symlinks...')
            count = 0
            for f in jpgs:
                src = os.path.join(dirpath, f)
                dst = os.path.join(faces_dir, f)
                if not os.path.exists(dst):
                    try:
                        os.symlink(src, dst)
                    except OSError:
                        shutil.copy2(src, dst)
                    count += 1
            print(f'Linked {count} images')
            celeba_found = True
            break

if not celeba_found:
    print('ERROR: Face images not found!')

# --- 3. Checkpoint ---
os.makedirs(CKPT_DIR, exist_ok=True)
dst_ckpt = os.path.join(CKPT_DIR, 'last.pth')
if not os.path.exists(dst_ckpt):
    src_ckpt = find_file(KAGGLE_INPUT, 'last.pth')
    if src_ckpt:
        shutil.copy2(src_ckpt, dst_ckpt)
        print(f'Checkpoint: {src_ckpt}')
    else:
        print('No last.pth found')
else:
    print(f'Checkpoint already at {dst_ckpt}')

# --- 4. Base model ---
base_dst = os.path.join(WORK_DIR, 'model.state_dict')
if not os.path.exists(base_dst):
    base_src = find_file(KAGGLE_INPUT, 'model.state_dict')
    if base_src:
        shutil.copy2(base_src, base_dst)
        print(f'Base model: {base_src}')

# --- 5. QuickDraw masks ---
masks_dir = os.path.join(DATASET_DIR, 'masks')
os.makedirs(masks_dir, exist_ok=True)
if not os.path.exists(os.path.join(masks_dir, 'face.ndjson')):
    !wget -q https://storage.googleapis.com/quickdraw_dataset/full/simplified/face.ndjson -P {masks_dir}
    print('Downloaded QuickDraw masks')
else:
    print('Masks already present')

# --- Verification ---
print('\n' + '='*60)
print('SETUP VERIFICATION')
print('='*60)
face_count = len(os.listdir(faces_dir)) if os.path.exists(faces_dir) else 0
has_train = os.path.exists(os.path.join(PROJECT_DIR, 'train.py'))
has_ckpt = os.path.exists(dst_ckpt)
has_base = os.path.exists(base_dst)
print(f'Face images:    {face_count}')
print(f'Masks:          {os.listdir(masks_dir)}')
print(f'Train.py:       {has_train}')
print(f'last.pth:       {has_ckpt}')
print(f'Base model:     {has_base}')
print()
if face_count > 1000 and has_train and (has_ckpt or has_base):
    print('ALL READY — proceed to Cell 3!')
else:
    if face_count <= 1000: print('  MISSING: face images')
    if not has_train: print('  MISSING: training code')
    if not has_ckpt and not has_base: print('  MISSING: checkpoint or base model')


In [ ]:
# Cell 3: Verify GPU
import torch

print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU 0: {props.name} ({props.total_mem / 1e9:.1f} GB)')
    print('\nNote: Using single GPU (this model\'s partial convolutions')
    print('are not compatible with DataParallel).')
    print('Training uses CUDA_VISIBLE_DEVICES=0 to avoid issues.')
else:
    print('ERROR: No GPU! Set Accelerator to T4 x2.')


In [ ]:
# Cell 4: Copy train.py into the project
import shutil, os

train_src = os.path.join(PROJECT_DIR, 'train.py')
train_dst = os.path.join(PROJECT_DIR, 'inpaint', 'train.py')

if os.path.exists(train_src):
    shutil.copy2(train_src, train_dst)
    print('Copied train.py to inpaint/')
    print('  + TensorBoard + W&B logging')
    print('  + Epoch tracking in checkpoints')
    print('  + Auto-resume from last.pth')
else:
    print('ERROR: train.py not found!')


In [ ]:
# Cell 5: Run Training (single GPU)
import os
os.chdir(os.path.join(PROJECT_DIR, 'inpaint'))

# Auto-detect checkpoint
last_ckpt = os.path.join(CKPT_DIR, 'last.pth')
base_model = os.path.join(WORK_DIR, 'model.state_dict')

if os.path.exists(last_ckpt):
    resume_from = last_ckpt
    print(f'Resuming from: {last_ckpt}')
elif os.path.exists(base_model):
    resume_from = base_model
    print(f'Starting from base model: {base_model}')
else:
    resume_from = ''
    print('WARNING: No checkpoint found — training from scratch!')

BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 3       # ~3.5 hrs/epoch × 3 = ~10.5 hrs (fits in 12hr session)
LR = 2e-5

resume_flag = f'--resume {resume_from}' if resume_from else ''

# CUDA_VISIBLE_DEVICES=0 forces single GPU (avoids DataParallel issues)
!CUDA_VISIBLE_DEVICES=0 python train.py \
  --train-dir {DATASET_DIR}/faces \
  --masks-dir {DATASET_DIR}/masks \
  {resume_flag} \
  --output-dir {CKPT_DIR} \
  --log-dir {TB_LOG_DIR} \
  --device cuda:0 \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --epochs {EPOCHS} \
  --lr {LR} \
  --no-amp


In [ ]:
# Cell 6: Check Saved Checkpoints
import os, torch

if os.path.exists(CKPT_DIR):
    files = sorted(os.listdir(CKPT_DIR))
    print(f'Checkpoints ({len(files)} files):')
    for f in files:
        size_mb = os.path.getsize(os.path.join(CKPT_DIR, f)) / (1024*1024)
        print(f'  {f} ({size_mb:.1f} MB)')

last_path = os.path.join(CKPT_DIR, 'last.pth')
if os.path.exists(last_path):
    data = torch.load(last_path, map_location='cpu', weights_only=False)
    if isinstance(data, dict) and 'epoch' in data:
        print(f'\nLast completed epoch: {data["epoch"]}')
        print(f'Next session starts at: epoch {data["epoch"] + 1}')


In [ ]:
# Cell 7: Package Checkpoints for Download
import shutil, os

if os.path.exists(CKPT_DIR) and len(os.listdir(CKPT_DIR)) > 0:
    archive = os.path.join(WORK_DIR, 'facerestore_checkpoints')
    shutil.make_archive(archive, 'zip', CKPT_DIR)
    size_mb = os.path.getsize(archive + '.zip') / (1024*1024)
    print(f'Created: facerestore_checkpoints.zip ({size_mb:.1f} MB)')
    print(f'\nNext steps:')
    print(f'  1. Download from Output tab')
    print(f'  2. Update checkpoints dataset with new last.pth')
    print(f'  3. Run notebook again — auto-resumes')
else:
    print('No checkpoints to package.')
